In [1]:
import ee
import geemap
import geopandas as gpd
import requests
import pandas as pd
import tempfile
import ipywidgets as widgets
from traitlets import link
from shapely.geometry import shape
from IPython.display import display



try:
    ee.Initialize(project='my-project-12126-484118')
    print("Earth Engine inicializado.")
except Exception as e:
    print("Autenticando Earth Engine...")
    ee.Authenticate()
    ee.Initialize(project='my-project-12126-484118')

print("\n--- PASO 1: DIBUJA TU LOTE ---")
print("Usa el polígono de la izquierda para dibujar. Cuando termines, pasa a la Celda 2.")
Draw_Map = geemap.Map(center=[33.584, -101.845], zoom=14)
Draw_Map.add_basemap('SATELLITE')
display(Draw_Map)

Earth Engine inicializado.

--- PASO 1: DIBUJA TU LOTE ---
Usa el polígono de la izquierda para dibujar. Cuando termines, pasa a la Celda 2.


Map(center=[33.584, -101.845], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

In [3]:
# ========================================
# 4. ANÁLISIS CLIMÁTICO (FECHAS AUTOMÁTICAS + ACUMULADO)
# ========================================
import ee
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

print("\n--- PASO 4: EXTRACCIÓN Y GRÁFICOS CLIMÁTICOS ---")

# 1. Definir punto de interés (CORREGIDO: Usando lote_geom con margen de error)
# 1. Verificación del dibujo
roi = Draw_Map.user_roi
if roi is None:
    raise ValueError("⚠️ No se detectó ningún dibujo. Volvé a la Celda 1, dibujá el lote y ejecutá esta celda.")

print("📐 Configurando geometría del lote...")
roi_info  = roi.getInfo()
lote_geom = ee.Geometry(roi_info['geometry'] if 'geometry' in roi_info else roi_info)
centro    = lote_geom.centroid(maxError=1)

# Usamos el centroide de la geometría sólida definida en el Paso 2
punto_clima = lote_geom.centroid(maxError=1)

# 2. Buscar la fecha más actual disponible en ERA5-Land
print("Buscando los datos climáticos más recientes en la Agencia Espacial...")
era5_base = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR").filterBounds(punto_clima)

# Tomamos la última imagen ingresada al catálogo
ultima_imagen = era5_base.sort('system:time_start', False).first()
fecha_max_ee = ultima_imagen.date().format('YYYY-MM-dd').getInfo()

# Calculamos los periodos automáticamente con Pandas
fecha_fin = pd.to_datetime(fecha_max_ee)

# Historico: 3 años hacia atras desde el mes actual
hist_start = (fecha_fin - pd.DateOffset(years=3)).replace(day=1).strftime('%Y-%m-%d')
hist_end = fecha_max_ee

# Diario: Desde el dia 1 del mes de la ultima fecha disponible hasta esa fecha
daily_start = fecha_fin.replace(day=1).strftime('%Y-%m-%d')
daily_end = fecha_max_ee

print(f"✅ Datos históricos: {hist_start} a {hist_end}")
print(f"✅ Datos diarios (Mes de análisis): {daily_start} a {daily_end}")

# 3. Extracción de datos de GEE
era5 = era5_base.filterDate(hist_start, ee.Date(hist_end).advance(1, 'day'))

def obtener_valores(img):
    fecha = img.date().format('YYYY-MM-dd')
    valores = img.reduceRegion(reducer=ee.Reducer.first(), geometry=punto_clima, scale=9000)

    # Conversiones: Kelvin a Celsius y Metros a Milímetros
    t_max = ee.Number(valores.get('temperature_2m_max')).subtract(273.15)
    t_min = ee.Number(valores.get('temperature_2m_min')).subtract(273.15)
    t_med = ee.Number(valores.get('temperature_2m')).subtract(273.15)
    precip = ee.Number(valores.get('total_precipitation_sum')).multiply(1000)

    return ee.Feature(None, {
        'fecha': fecha,
        't_max': t_max,
        't_min': t_min,
        't_med': t_med,
        'precip': precip
    })

print("Descargando serie temporal (esto puede tardar unos 20-30 segundos)...")
# Filtramos nulos para asegurar que la serie sea continua y limpia
datos_ee = era5.map(obtener_valores).filter(ee.Filter.notNull(['t_med'])).getInfo()['features']

# 4. Procesamiento con Pandas
df = pd.DataFrame([d['properties'] for d in datos_ee])
df['fecha'] = pd.to_datetime(df['fecha'])

# --- DataFrame Mensual ---
df_mensual = df.copy()
df_mensual['mes_año'] = df_mensual['fecha'].dt.to_period('M')
resumen_mensual = df_mensual.groupby('mes_año').agg({
    't_max': 'mean',
    't_min': 'mean',
    'precip': 'sum'
}).reset_index()
resumen_mensual['mes_año'] = resumen_mensual['mes_año'].dt.to_timestamp()

# --- DataFrame Diario (Zoom al mes actual) ---
df_diario = df[(df['fecha'] >= daily_start) & (df['fecha'] <= daily_end)].copy()
df_diario = df_diario.sort_values('fecha')

# ¡NUEVO!: Calculamos el acumulado de lluvia del mes
df_diario['precip_acum'] = df_diario['precip'].cumsum()

# 5. Gráficos Interactivos (Plotly)
print("Generando paneles interactivos...")

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=False,
    vertical_spacing=0.15,
    subplot_titles=(f"<b>Histórico Mensual</b> (Últimos 2 años)",
                    f"<b>Detalle Diario</b> ({daily_start} al {daily_end})"),
    specs=[[{"secondary_y": True}], [{"secondary_y": True}]]
)

# --- PANEL 1: MENSUAL ---
fig.add_trace(go.Bar(x=resumen_mensual['mes_año'], y=resumen_mensual['precip'],
                     name='Lluvia Mensual (mm)', marker_color='#3498db', opacity=0.8),
              row=1, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=resumen_mensual['mes_año'], y=resumen_mensual['t_max'],
                         name='T. Máx Media (°C)', mode='lines+markers', line=dict(color='#e74c3c')),
              row=1, col=1, secondary_y=True)
fig.add_trace(go.Scatter(x=resumen_mensual['mes_año'], y=resumen_mensual['t_min'],
                         name='T. Mín Media (°C)', mode='lines+markers', line=dict(color='#2c3e50')),
              row=1, col=1, secondary_y=True)

# --- PANEL 2: DIARIO ---
fig.add_trace(go.Bar(x=df_diario['fecha'], y=df_diario['precip'],
                     name='Lluvia Diaria (mm)', marker_color='#2980b9'),
              row=2, col=1, secondary_y=False)

# Línea de Lluvia Acumulada
fig.add_trace(go.Scatter(x=df_diario['fecha'], y=df_diario['precip_acum'],
                         name='Lluvia Acumulada (mm)', mode='lines', line=dict(color='#9b59b6', width=3, dash='dot')),
              row=2, col=1, secondary_y=False)

fig.add_trace(go.Scatter(x=df_diario['fecha'], y=df_diario['t_med'],
                         name='T. Media Diaria (°C)', mode='lines', line=dict(color='#e67e22', width=3)),
              row=2, col=1, secondary_y=True)

# --- Diseño y Layout ---
fig.update_layout(
    height=850,
    template='plotly_white',
    title_text="🌦️ Climatología del Lote (Fuente: ERA5-Land)",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.update_yaxes(title_text="Precipitación (mm)", secondary_y=False, row=1, col=1, showgrid=False)
fig.update_yaxes(title_text="Temperatura (°C)", secondary_y=True, row=1, col=1, showgrid=True)
fig.update_yaxes(title_text="Precipitación (mm)", secondary_y=False, row=2, col=1, showgrid=False)
fig.update_yaxes(title_text="Temperatura (°C)", secondary_y=True, row=2, col=1, showgrid=True)

display(fig)




--- PASO 4: EXTRACCIÓN Y GRÁFICOS CLIMÁTICOS ---
📐 Configurando geometría del lote...
Buscando los datos climáticos más recientes en la Agencia Espacial...
✅ Datos históricos: 2023-04-01 a 2026-04-07
✅ Datos diarios (Mes de análisis): 2026-04-01 a 2026-04-07
Descargando serie temporal (esto puede tardar unos 20-30 segundos)...
Generando paneles interactivos...
